# Context Engineering and Dense Vector Retrieval

An LLM answers from what is in its context window. **Context engineering** is
the discipline of deciding what goes in there — and retrieval is how we find
it. This session builds the intuition bottom-up: why keyword search is not
enough, what an embedding is, and how similarity search actually ranks
documents.

Everything runs offline and deterministically, so you can focus on the
mechanics rather than API plumbing.

## Why stuffing the context fails

The naive move is to paste the entire knowledge base into the prompt. That
fails three ways: context windows are finite, attention degrades over long
irrelevant spans, and you pay per token for text the model did not need.
So the real question is *selection*: given a query, which few passages
deserve the window?

In [1]:
# The knowledge base for every lab this week: support documents for Atlas
# Cycles, a fictional e-bike maker. Small enough to read, real enough to
# retrieve against.
CORPUS = {
    "battery-care": (
        "Atlas S2 battery care. Charge the battery to 80 percent for daily "
        "use and only to 100 percent before a long ride. Store between 10 "
        "and 25 degrees Celsius. A full recharge takes 4.5 hours from empty."
    ),
    "warranty": (
        "Atlas warranty policy. The frame is covered for 5 years. The "
        "battery and motor are covered for 2 years or 15,000 km, whichever "
        "comes first. Wear parts such as brake pads and tires are excluded."
    ),
    "range": (
        "Atlas S2 range guide. Expect 90 to 110 km in Eco mode, 60 to 75 km "
        "in Trail mode, and 40 to 55 km in Boost mode. Headwind, cargo "
        "weight, and cold weather reduce range by up to 30 percent."
    ),
    "error-codes": (
        "Atlas display error codes. E01 means a motor sensor fault: restart "
        "the system. E04 means battery communication lost: reseat the "
        "battery. E09 means brake cutoff engaged: check the brake levers."
    ),
    "first-service": (
        "First service. Book the complimentary first service after 300 km "
        "or 3 months. Spoke tension, brake bedding, and firmware updates "
        "are included at no charge."
    ),
}

print(f"{len(CORPUS)} documents, "
      f"{sum(len(v.split()) for v in CORPUS.values())} words total")

5 documents, 167 words total


## Keyword search breaks on paraphrase

A learner asks: *"how long does a full charge take?"* The battery document
answers it — but says "recharge", not "charge". Exact-match search misses
what a human plainly sees.

In [2]:
query = "how long does a full charge take"

def keyword_score(query, doc):
    q, d = set(query.lower().split()), set(doc.lower().split())
    return len(q & d)

for name, doc in sorted(CORPUS.items(),
                        key=lambda kv: -keyword_score(query, kv[1])):
    print(f"{keyword_score(query, doc):>2}  {name}")

 4  battery-care
 1  error-codes
 0  warranty
 0  range
 0  first-service


The overlap words ("a", "how", "long"…) are noise — `range` ties or beats
`battery-care` depending on stop words. The signal we want is *meaning*
overlap, not string overlap.

## Embeddings: text as geometry

An embedding model maps text to a vector such that similar meanings land
near each other. We will fake the *model* with character trigrams — but the
**geometry is the real thing**: ranking by cosine similarity between vectors
is exactly what production retrieval does.

In [3]:
# A deterministic embedding: character trigram counts. No model, no network,
# yet it captures enough word-shape overlap to demonstrate the geometry that
# real embedding models learn.
from collections import Counter
import math

def embed(text):
    t = " " + "".join(c.lower() if c.isalnum() else " " for c in text) + " "
    return Counter(t[i:i+3] for i in range(len(t) - 2) if t[i:i+3].strip())

def cosine(a, b):
    dot = sum(a[k] * b[k] for k in a.keys() & b.keys())
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

doc_vectors = {name: embed(doc) for name, doc in CORPUS.items()}
q = embed(query)
for name, vec in sorted(doc_vectors.items(),
                        key=lambda kv: -cosine(q, kv[1])):
    print(f"{cosine(q, vec):.3f}  {name}")

0.318  battery-care
0.162  first-service
0.092  error-codes
0.063  warranty
0.030  range


`battery-care` now wins decisively: "recharge" and "charge" share most of
their trigrams, so the vectors overlap even though the exact word differs.
That is the entire trick of dense retrieval, in miniature.

## Top-k retrieval

Production systems return the *k* best passages with scores, and everything
downstream — RAG, agents, reranking — consumes that ranked list.

In [4]:
def retrieve(query, k=2):
    q = embed(query)
    ranked = sorted(((cosine(q, v), name) for name, v in doc_vectors.items()),
                    reverse=True)
    return ranked[:k]

for question in [
    "is the motor still under warranty after 20,000 km",
    "screen shows E04, what do I do",
    "when should I bring the bike in for its first checkup",
]:
    print(question)
    for score, name in retrieve(question):
        print(f"   {score:.3f}  {name}")

is the motor still under warranty after 20,000 km
   0.292  warranty
   0.158  first-service
screen shows E04, what do I do
   0.059  error-codes
   0.031  battery-care
when should I bring the bike in for its first checkup
   0.242  first-service
   0.221  warranty


## What changes in production

Three upgrades, same shape: a **learned embedding model** replaces trigrams,
an **approximate nearest-neighbor index** (HNSW, IVF) replaces the linear
scan, and documents are **chunked** so a passage — not a whole file — is the
unit of retrieval. You will build that pipeline end to end in the next
session.

**Takeaways**
- Context engineering is selection: the model answers from the window you build.
- Keyword overlap breaks on paraphrase; vector similarity survives it.
- Retrieval returns a ranked, scored list — every later pattern consumes it.